In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader

from tqdm.auto import tqdm

from dfm.data.salinas import (
    SALINAS_CLASS_NAMES,
    SalinasPatchDataset,
    load_salinas,
)

from dfm.models.cnn_baseline import CNNBaseline

from dfm.training.metrics import (
    accuracy_score,
    macro_f1_score,
)

from dfm.training.profiling import count_parameters

c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

data_dir = PROJECT_ROOT / "data" / "raw" / "salinas"

scene = load_salinas(
    data_dir,
    download=True,
)

print("Cube shape:", scene.cube.shape)
print("Label map shape:", scene.labels.shape)
print("Bands:", scene.bands)
print("Classes:", len(scene.class_names))

Cube shape: (512, 217, 204)
Label map shape: (512, 217)
Bands: 204
Classes: 16


In [4]:
split_path = outputs_dir / "salinas_spatial_split_seed42.npz"

split = np.load(split_path)

train_indices = split["train_indices"]
val_indices = split["val_indices"]
test_indices = split["test_indices"]

print("Train samples:", len(train_indices))
print("Validation samples:", len(val_indices))
print("Test samples:", len(test_indices))

Train samples: 32337
Validation samples: 10952
Test samples: 10840


In [5]:
patch_size = 15

train_dataset = SalinasPatchDataset(
    scene,
    indices=train_indices,
    patch_size=patch_size,
)

val_dataset = SalinasPatchDataset(
    scene,
    indices=val_indices,
    patch_size=patch_size,
)

test_dataset = SalinasPatchDataset(
    scene,
    indices=test_indices,
    patch_size=patch_size,
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Train samples: 32337
Validation samples: 10952
Test samples: 10840


In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

In [7]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = CNNBaseline(
    in_channels=scene.bands,
    num_classes=len(SALINAS_CLASS_NAMES),
).to(device)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print(
    "Trainable parameters:",
    count_parameters(model)
)

Device: cuda
GPU: NVIDIA GeForce RTX 2050
Trainable parameters: 126480


In [8]:
x, y = next(iter(train_loader))

x = x.to(
    device=device,
    dtype=torch.float32,
)

with torch.no_grad():
    output = model(x)

print("Input:", x.shape)
print("Output:", output.shape)

Input: torch.Size([128, 204, 15, 15])
Output: torch.Size([128, 16])


In [11]:
model = CNNBaseline(
    in_channels=scene.bands,
    num_classes=len(SALINAS_CLASS_NAMES),
).to(device)

print(model)

print(
    "Trainable parameters:",
    count_parameters(model)
)

CNNBaseline(
  (features): Sequential(
    (0): Conv2d(204, 64, kernel_size=(1, 1), stride=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): GELU(approximate='none')
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): GELU(approximate='none')
    (9): AdaptiveAvgPool2d(output_size=1)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.1, inplace=False)
    (2): Linear(in_features=128, out_features=16, bias=True)
  )
)
Trainable parameters: 126480


In [12]:
x, y = next(iter(train_loader))

print("Original batch:", x.shape)
print("Labels:", y.shape)

x = x.to(
    device=device,
    dtype=torch.float32,
)

with torch.no_grad():
    output = model(x)

print("Model output:", output.shape)

Original batch: torch.Size([128, 204, 15, 15])
Labels: torch.Size([128])
Model output: torch.Size([128, 16])


In [13]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

print("Loss:", criterion)
print("Optimizer:", optimizer)

Loss: CrossEntropyLoss()
Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.0001
)


In [14]:
def train_one_epoch(model, loader):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for x, y in tqdm(
        loader,
        desc="Training",
        leave=False,
    ):

        x = x.to(
            device=device,
            dtype=torch.float32,
        )

        y = y.to(
            device=device,
            dtype=torch.long,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(x)

        loss = criterion(
            logits,
            y,
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item() * x.shape[0]
        )

        total_samples += x.shape[0]

    return total_loss / total_samples

In [15]:
@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    all_true = []
    all_pred = []

    for x, y in tqdm(
        loader,
        desc="Validation",
        leave=False,
    ):

        x = x.to(
            device=device,
            dtype=torch.float32,
        )

        logits = model(x)

        pred = (
            logits
            .argmax(dim=1)
            .cpu()
            .numpy()
        )

        all_pred.append(pred)
        all_true.append(
            y.numpy()
        )

    y_true = np.concatenate(
        all_true
    )

    y_pred = np.concatenate(
        all_pred
    )

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),

        "macro_f1": macro_f1_score(
            y_true,
            y_pred,
            num_classes=len(
                SALINAS_CLASS_NAMES
            ),
        ),
    }

In [16]:
test_loss = train_one_epoch(
    model,
    train_loader,
)

test_metrics = evaluate(
    model,
    val_loader,
)

print("Test training loss:", test_loss)
print("Test validation metrics:", test_metrics)

Test training loss: 0.3084665833148037
Test validation metrics: {'accuracy': 0.9064097881665449, 'macro_f1': 0.9149211982517331}


In [17]:
epochs = 30
patience = 8

best_macro_f1 = -1.0
best_epoch = None
best_state = None

epochs_without_improvement = 0

history = []

cnn_checkpoint_path = (
    outputs_dir / "cnn_spatial_best.pt"
)

print(
    "Checkpoint:",
    cnn_checkpoint_path
)

Checkpoint: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\cnn_spatial_best.pt


In [18]:
for epoch in range(1, epochs + 1):

    train_loss = train_one_epoch(
        model,
        train_loader,
    )

    val_metrics = evaluate(
        model,
        val_loader,
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Macro-F1: {val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > best_macro_f1:

        best_macro_f1 = (
            val_metrics["macro_f1"]
        )

        best_epoch = epoch

        epochs_without_improvement = 0

        # Copy best weights to CPU memory
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

        # Save immediately
        torch.save(
            {
                "model_state_dict": best_state,
                "best_epoch": best_epoch,
                "best_val_macro_f1": best_macro_f1,
                "class_names": SALINAS_CLASS_NAMES,
                "patch_size": patch_size,
                "bands": scene.bands,
                "seed": SEED,
            },
            cnn_checkpoint_path,
        )

        print(
            "✓ New best CNN model — checkpoint saved"
        )

    else:

        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:

        print(
            f"Early stopping at epoch {epoch}"
        )

        break

Epoch 01 | Loss: 0.0483 | Val Acc: 0.7385 | Val Macro-F1: 0.7646
✓ New best CNN model — checkpoint saved


Epoch 02 | Loss: 0.0360 | Val Acc: 0.8430 | Val Macro-F1: 0.9013
✓ New best CNN model — checkpoint saved


Epoch 03 | Loss: 0.0305 | Val Acc: 0.8719 | Val Macro-F1: 0.8840


Epoch 04 | Loss: 0.0282 | Val Acc: 0.8692 | Val Macro-F1: 0.9003


Epoch 05 | Loss: 0.0161 | Val Acc: 0.8168 | Val Macro-F1: 0.8890


...

Epoch 26 | Loss: 0.0049 | Val Acc: 0.8645 | Val Macro-F1: 0.8870


Epoch 27 | Loss: 0.0063 | Val Acc: 0.8387 | Val Macro-F1: 0.9113


Epoch 28 | Loss: 0.0035 | Val Acc: 0.8892 | Val Macro-F1: 0.9465


Epoch 29 | Loss: 0.0030 | Val Acc: 0.8482 | Val Macro-F1: 0.9149


Epoch 30 | Loss: 0.0047 | Val Acc: 0.8888 | Val Macro-F1: 0.9454


In [19]:
cnn_checkpoint_path = outputs_dir / "cnn_spatial_best.pt"

cnn_checkpoint = torch.load(
    cnn_checkpoint_path,
    map_location="cpu",
)

print("Best epoch:", cnn_checkpoint["best_epoch"])
print(
    "Best validation Macro-F1:",
    cnn_checkpoint["best_val_macro_f1"]
)
print(
    "Weights type:",
    type(cnn_checkpoint["model_state_dict"])
)
print(
    "Number of tensors:",
    len(cnn_checkpoint["model_state_dict"])
)

Best epoch: 23
Best validation Macro-F1: 0.9675768765001878
Weights type: <class 'dict'>
Number of tensors: 23


C:\Users\Dines\AppData\Local\Temp\ipykernel_29656\2517707274.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_checkpoint = torch.load(


In [20]:
model.load_state_dict(
    cnn_checkpoint["model_state_dict"]
)

model.to(device)
model.eval()

print("Best CNN checkpoint restored.")
print(
    "Epoch:",
    cnn_checkpoint["best_epoch"]
)
print(
    "Validation Macro-F1:",
    cnn_checkpoint["best_val_macro_f1"]
)

Best CNN checkpoint restored.
Epoch: 23
Validation Macro-F1: 0.9675768765001878


In [21]:
cnn_test_metrics = evaluate(
    model,
    test_loader,
)

print("=" * 60)
print("FINAL CNN SPATIAL TEST RESULTS")
print("=" * 60)

print(
    f"Accuracy : "
    f"{cnn_test_metrics['accuracy']:.4f}"
)

print(
    f"Macro-F1 : "
    f"{cnn_test_metrics['macro_f1']:.4f}"
)

FINAL CNN SPATIAL TEST RESULTS
Accuracy : 0.9601
Macro-F1 : 0.9415


In [22]:
@torch.no_grad()
def collect_predictions(model, loader):

    model.eval()

    all_true = []
    all_pred = []

    for x, y in tqdm(
        loader,
        desc="Collecting CNN test predictions",
    ):

        x = x.to(
            device=device,
            dtype=torch.float32,
        )

        logits = model(x)

        pred = (
            logits
            .argmax(dim=1)
            .cpu()
            .numpy()
        )

        all_pred.append(pred)
        all_true.append(
            y.numpy()
        )

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)

    return y_true, y_pred

In [23]:
y_test_cnn, y_pred_cnn = collect_predictions(
    model,
    test_loader,
)

print("Test samples:", len(y_test_cnn))
print("Predictions:", len(y_pred_cnn))

Test samples: 10840
Predictions: 10840


In [24]:
np.save(
    outputs_dir / "cnn_spatial_y_test.npy",
    y_test_cnn,
)

np.save(
    outputs_dir / "cnn_spatial_y_pred.npy",
    y_pred_cnn,
)

print("CNN predictions saved.")

CNN predictions saved.


In [25]:
cnn_results = {
    "model": "CNN",
    "best_epoch": cnn_checkpoint["best_epoch"],
    "val_macro_f1": cnn_checkpoint["best_val_macro_f1"],
    "test_accuracy": cnn_test_metrics["accuracy"],
    "test_macro_f1": cnn_test_metrics["macro_f1"],
    "parameters": count_parameters(model),
}

cnn_results_df = pd.DataFrame([cnn_results])

cnn_results_df.to_csv(
    outputs_dir / "cnn_spatial_results.csv",
    index=False,
)

cnn_results_df

,model,best_epoch,val_macro_f1,test_accuracy,test_macro_f1,parameters
0,CNN,23,0.967577,0.960148,0.94145,126480


In [26]:
comparison_df = pd.DataFrame([
    {
        "model": "CNN",
        "test_accuracy": cnn_test_metrics["accuracy"],
        "test_macro_f1": cnn_test_metrics["macro_f1"],
    },
    {
        "model": "Hybrid Spatial-Spectral",
        "test_accuracy": 0.9532,
        "test_macro_f1": 0.9206,
    },
])

comparison_df

,model,test_accuracy,test_macro_f1
0,CNN,0.960148,0.94145
1,Hybrid Spatial-Spectral,0.953200,0.92060


In [27]:
cnn_acc = cnn_test_metrics["accuracy"]
cnn_f1 = cnn_test_metrics["macro_f1"]

hybrid_acc = 0.9532
hybrid_f1 = 0.9206

print(
    f"CNN vs Hybrid Accuracy difference: "
    f"{(cnn_acc - hybrid_acc) * 100:.2f} pp"
)

print(
    f"CNN vs Hybrid Macro-F1 difference: "
    f"{(cnn_f1 - hybrid_f1) * 100:.2f} pp"
)

CNN vs Hybrid Accuracy difference: 0.69 pp
CNN vs Hybrid Macro-F1 difference: 2.09 pp
